In [4]:
from peft import PeftModel, PeftConfig
from transformers import AutoTokenizer, AutoModelForCausalLM

import torch

In [ ]:
# Load adapter config to get base model
peft_config = PeftConfig.from_pretrained("outputs-full-sft-v1/checkpoint-1077")
base_model = AutoModelForCausalLM.from_pretrained(peft_config.base_model_name_or_path)
tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path)

# Merge adapter
model = PeftModel.from_pretrained(base_model, "outputs-full-sft-v1/checkpoint-1077")
model.eval()


/home/jack/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-26 14:31:26.403013: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-26 14:31:26.466571: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-26 14:31:27.427186: W tensorflow/compiler/tf2tensorrt/utils/py_uti

[2025-05-26 14:31:28,779] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/jack/anaconda3/envs/llava/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/jack/anaconda3/envs/llava/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda-11.8/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/jack/anaconda3/envs/llava/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda-11.8/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/jack/anaconda3/envs/llava/compiler_compat/ld: /usr/local/cuda-11.8/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/home/jack/anaconda3/envs/llava/compiler_compat/ld: /usr/local/cuda-11.8/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3'
/home/jack/anaconda3/envs/llava/compiler_compat/ld: /usr/local/cuda-11.8/lib64/libcufile.so: undefined reference to `std::ostream::tellp()@GLIBCXX_3.4'
/home/jack/anaconda3/envs/llava/compiler_compat/ld: 

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): Linear(
                in_features=896, out_features=896, bias=True
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
              (k_proj): Linear(
                in_features=896, out_features=128, bias=True
                (lora_dropout):

In [15]:
prompt = "Produce the answer plus a minimal justification referencing the evidence provided.\n\n### Question:\nAre routine preoperative restaging CTs after neoadjuvant chemoradiation for locally advanced rectal cancer low yield : a retrospective case study?\n\n### Context:\nPre-operative restaging CT scans are often performed routinely following neoadjuvant chemoradiotherapy for locally advanced rectal cancer. There is a paucity of data on the utility of this common practice. We sought to determine how often restaging CTs identified disease progression or regression that altered management.\n\nWe performed a single-institution retrospective study. From 2007 to 2011, 182 patients had newly-diagnosed, non-metastatic rectal adenocarcinoma, of which 96 were surgical candidates with clinical stage II/III disease. Ninety-one of these patients (95%) completed neoadjuvant chemoradiation.\n\nEighty-three out of 91 patients (91%) had restaging CTs. Four patients (5%) had new lesions suspicious for distant metastasis (2 lung, 2 liver) on restaging CT scan reports (1 of these was present on initial staging CT but not reported). All 4 patients had node-positive disease. In no case did restaging CT result in a change in surgical management."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

In [17]:
with torch.no_grad():
    output = model.base_model.generate(input_ids=input_ids, max_new_tokens=4096, do_sample=False)

response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)

Produce the answer plus a minimal justification referencing the evidence provided.

### Question:
Are routine preoperative restaging CTs after neoadjuvant chemoradiation for locally advanced rectal cancer low yield : a retrospective case study?

### Context:
Pre-operative restaging CT scans are often performed routinely following neoadjuvant chemoradiotherapy for locally advanced rectal cancer. There is a paucity of data on the utility of this common practice. We sought to determine how often restaging CTs identified disease progression or regression that altered management.

We performed a single-institution retrospective study. From 2007 to 2011, 182 patients had newly-diagnosed, non-metastatic rectal adenocarcinoma, of which 96 were surgical candidates with clinical stage II/III disease. Ninety-one of these patients (95%) completed neoadjuvant chemoradiation.

Eighty-three out of 91 patients (91%) had restaging CTs. Four patients (5%) had new lesions suspicious for distant metastasi

In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import json
import re

# -------------------------
# 1. Load base + adapter model
# -------------------------
base_model_path = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_path = "outputs-full-sft-v1/checkpoint-800"

tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(base_model_path, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

# -------------------------
# 2. Format input prompt
# -------------------------
instruction = "Apply multi-scale super-resolution to a CT image of the chest with pulmonary embolism."
prompt = f"<|user|>\n{instruction}\n<|assistant|>\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# -------------------------
# 3. Generate output
# -------------------------
with torch.no_grad():
    output = model.generate(
        input_ids=inputs["input_ids"],
        max_new_tokens=1024,
        do_sample=False,
        return_dict_in_generate=True,
        output_scores=True
    )

# Decode full output (don't skip special tokens)
generated_text = tokenizer.decode(output.sequences[0], skip_special_tokens=False)
print("----- FULL OUTPUT -----")
print(generated_text)

# -------------------------
# 4. Try extracting JSON tool-call
# -------------------------
print("\n----- PARSED TOOL CALL -----")
matches = re.findall(r'\{.*"API_name".*?\}', generated_text, re.DOTALL)
if matches:
    try:
        tool_call = json.loads(matches[0])
        print(json.dumps(tool_call, indent=2))
    except json.JSONDecodeError:
        print("Found JSON-like string but couldn't decode it.")
        print(matches[0])
else:
    print("No tool-call structure found in output.")


/home/jack/.local/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jack/.local/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jack/.local/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


----- FULL OUTPUT -----
<|user|>
Apply multi-scale super-resolution to a CT image of the chest with pulmonary embolism.
<|assistant|>
Calling HealthGPT super_resolution...<|im_end|>

----- PARSED TOOL CALL -----
No tool-call structure found in output.
